# El Niño 2026 — Notebook 03: Why did 2026 become historic?
**le-gis.com climate project · synthesis**

Notebooks 01–02 showed the mechanism for 2026. This notebook asks the comparative question: **which ingredients were stronger in 2026 than in the great El Niños of 1997–98 and 2015–16, and the recent 2023–24 event?**

| Ingredient | Hypothesis | Metric (this notebook) | Data |
|---|---|---|---|
| Recharge | H1 | Pre-onset equatorial heat content: basin-mean D20 anomaly (120°E–80°W), Jan Y-1 → Mar Y | GLORYS12 (cached in Notebook 02) |
| Trigger | H2 | Westerly wind bursts Nov Y-1 → Jun Y: count, days, peak, accumulated westerly forcing | ERA5 10 m wind (cached) |
| Coupling / peak | H3 | Peak Niño-3.4, Niño-1+2, ONI | OISST via Earth Engine, 1991–2020 base |
| Warm baseline | H4 | ONI − RONI (how much is global ocean warming) | OISST: Niño-3.4 minus 20°S–20°N mean |
| Indian Ocean | IOD | Dipole Mode Index, Sep–Nov | OISST (Saji et al. 1999 boxes) |
| Impact | Indonesia | Jun → mid-Oct rainfall anomaly (%) vs 1991–2020 | CHIRPS |

Events: **1997** (onset year Y = 1997), **2015**, **2023**, **2026**. Values for 2026 are *to date* (the event has not peaked).

Note: NOAA OISST `anom` is relative to 1971–2000; everything here is rebased to **1991–2020** (WMO/NOAA/BMKG standard).

In [ ]:
!pip -q install "cdsapi>=0.7.4" copernicusmarine earthengine-api geemap netCDF4
import os, json, time, getpass, threading, logging, warnings
import numpy as np, pandas as pd, xarray as xr, matplotlib.pyplot as plt, requests, io
warnings.filterwarnings("ignore"); logging.getLogger("copernicusmarine").setLevel(logging.ERROR)
plt.rcParams.update({"axes.grid": True, "grid.alpha": .25, "font.size": 9})

from google.colab import drive
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/le-gis_enso"
CACHE = f"{ROOT}/cache"; WEB = f"{ROOT}/web"
os.makedirs(CACHE, exist_ok=True); os.makedirs(WEB, exist_ok=True)

TODAY = pd.Timestamp.today().normalize(); this_year = TODAY.year
LON0, LON1, ZMAX = 120, 280, 400
EVENTS = [1997, 2015, 2023, 2026]
COLORS = {1997: "#1f77b4", 2015: "#2ca02c", 2023: "#9467bd", 2026: "crimson"}

def clean_time(da):
    da = da.sortby("time")
    _, keep = np.unique(da.time.values, return_index=True)
    return da.isel(time=np.sort(keep))

def rebase_monthly(df, y0=1991, y1=2020):
    # Remove the y0-y1 mean of each calendar month (anomalies -> 1991-2020 base).
    base = df[(df.index.year >= y0) & (df.index.year <= y1)]
    clim = base.groupby(base.index.month).mean()
    return df - clim.reindex(df.index.month).values

In [ ]:
# --- Credentials (needed only for data not yet cached: ERA5 2021-2024, GLORYS 2021-2026) ---
if not os.path.exists(os.path.expanduser("~/.cdsapirc")):
    token = getpass.getpass("CDS Personal Access Token: ")
    open(os.path.expanduser("~/.cdsapirc"), "w").write(f"url: https://cds.climate.copernicus.eu/api\nkey: {token}\n")
import copernicusmarine as cm
cm.login(username=input("Copernicus Marine username: "),
         password=getpass.getpass("Copernicus Marine password: "), force_overwrite=True)
import ee
PROJECT = "YOUR-EE-PROJECT"   # <-- change
ee.Authenticate(); ee.Initialize(project=PROJECT)

## A. Monthly SST indices 1982–2026 from OISST (Earth Engine), 1991–2020 base
Niño-3.4, Niño-1+2, tropical mean (20°S–20°N), IOD west (50–70°E, 10°S–10°N) and east (90–110°E, 10°S–0°).
**RONI** follows NOAA CPC's recipe: (Niño-3.4 − tropical mean), 3-month mean, rescaled to the variance of ONI.
One Earth Engine request per year, cached to Drive.

In [ ]:
oisst = ee.ImageCollection("NOAA/CDR/OISST/V2_1").select("anom")
BOXES = {"nino34": [-170, -5, -120, 5], "nino12": [-90, -10, -80, 0], "trop": [-180, -20, 180, 20],
         "iod_w": [50, -10, 70, 10], "iod_e": [90, -10, 110, 0]}
REG = ee.FeatureCollection([ee.Feature(ee.Geometry.BBox(*b), {"name": k}) for k, b in BOXES.items()])

last_day = pd.Timestamp(oisst.filterDate(str((TODAY - pd.Timedelta(days=40)).date()), str(TODAY.date()))
                        .aggregate_max("system:time_start").getInfo(), unit="ms").normalize()
last_full_month = (last_day + pd.Timedelta(days=1)).to_period("M").to_timestamp() - pd.offsets.MonthBegin(1)
print("OISST to", last_day.date(), "| last complete month:", f"{last_full_month:%Y-%m}")

def year_indices(y):
    f = f"{CACHE}/oisst_boxes_{y}.json"
    if os.path.exists(f) and y < this_year:
        return json.load(open(f))
    m_last = 12 if y < last_full_month.year else last_full_month.month
    fcs = []
    for m in range(1, m_last + 1):
        s = ee.Date.fromYMD(y, m, 1)
        img = oisst.filterDate(s, s.advance(1, "month")).mean().multiply(0.01).rename("anom")
        fc = img.reduceRegions(collection=REG, reducer=ee.Reducer.mean(), scale=27830, tileScale=4)
        fcs.append(fc.map(lambda ft, m=m: ft.set("m", m)))
    out = [{"y": y, "m": f["properties"]["m"], "name": f["properties"]["name"], "v": f["properties"].get("mean")}
           for f in ee.FeatureCollection(fcs).flatten().getInfo()["features"]]
    if y < this_year:
        json.dump(out, open(f, "w"))
    return out

rows = []
for y in range(1982, last_full_month.year + 1):
    rows += year_indices(y)
    if y % 5 == 0: print("  OISST boxes through", y)
raw = pd.DataFrame(rows)
raw["date"] = pd.to_datetime(dict(year=raw.y, month=raw.m, day=1))
sst71 = raw.pivot(index="date", columns="name", values="v").sort_index()      # anomalies vs 1971-2000
idx = rebase_monthly(sst71)                                                   # -> 1991-2020

idx["dmi"] = idx.iod_w - idx.iod_e
idx["oni"] = idx.nino34.rolling(3, center=True).mean()
rel3 = (idx.nino34 - idx.trop).rolling(3, center=True).mean()
b = slice("1991", "2020")
idx["roni"] = rel3 * idx.oni[b].std() / rel3[b].std()
idx.to_csv(f"{WEB}/indices_monthly_oisst_9120.csv", index_label="date")
idx.tail(6).round(2)

In [ ]:
# Cross-check against NOAA CPC's official ONI (ERSSTv5) - should agree closely
SEAS = {s: i + 1 for i, s in enumerate("DJF JFM FMA MAM AMJ MJJ JJA JAS ASO SON OND NDJ".split())}
try:
    t = pd.read_csv(io.StringIO(requests.get("https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt",
                                             timeout=60).text), sep=r"\s+")
    t.index = pd.to_datetime(dict(year=t.YR, month=t.SEAS.map(SEAS), day=1))
    cpc_oni = t["ANOM"]
    j = pd.concat([idx.oni, cpc_oni.rename("cpc")], axis=1).dropna()
    print(f"Our ONI vs CPC ONI: r = {j.corr().iloc[0,1]:.3f}, mean diff = {(j.oni - j.cpc).mean():+.2f} °C, "
          f"RMS diff = {np.sqrt(((j.oni - j.cpc)**2).mean()):.2f} °C  (n={len(j)})")
except Exception as e:
    cpc_oni = None; print("CPC ONI not fetched:", e)

fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(idx.index, idx.oni, 0, where=idx.oni > 0, color="#f4a582", alpha=.6, label="ONI (raw)")
ax.fill_between(idx.index, idx.oni, 0, where=idx.oni < 0, color="#92c5de", alpha=.6)
ax.plot(idx.index, idx.roni, c="k", lw=1.3, label="RONI (relative to tropical mean)")
if cpc_oni is not None:
    ax.plot(cpc_oni["1982":].index, cpc_oni["1982":], c="grey", lw=.8, ls=":", label="CPC ONI (check)")
for th in (0.5, 1.5, 2.5): ax.axhline(th, c="k", lw=.5, ls="--")
ax.set_title("ONI vs RONI, 1982–2026 (1991–2020 base) — the gap is the background ocean warming")
ax.set_ylabel("°C"); ax.legend(loc="upper left"); plt.show()

## B. Recharge (H1): equatorial heat content before onset — GLORYS12
Basin-mean D20 anomaly along 2°S–2°N, 120°E–80°W — a proxy for NOAA PMEL's Warm Water Volume.
Years 1993–2020 load from the Notebook 02 cache; 2021–2026 are fetched once (~1 min/year).

In [ ]:
GLORYS = [("cmems_mod_glo_phy_my_0.083deg_P1M-m", "reanalysis"),
          ("cmems_mod_glo_phy_myint_0.083deg_P1M-m", "interim"),
          ("cmems_mod_glo_phy-thetao_anfc_0.083deg_P1M-m", "analysis")]

def month_start(t):
    return pd.to_datetime(np.atleast_1d(t)).to_period("M").to_timestamp()

def cm_open(ds_id, lon_a, lon_b, t0=None, t1=None):
    kw = dict(dataset_id=ds_id, variables=["thetao"], minimum_longitude=lon_a, maximum_longitude=lon_b,
              minimum_latitude=-2, maximum_latitude=2, minimum_depth=0, maximum_depth=ZMAX)
    if t0 is not None:
        kw.update(start_datetime=str(t0.date()), end_datetime=str(t1.date()))
    return cm.open_dataset(**kw)["thetao"]

COVER = {}
for ds_id, label in GLORYS:
    try:
        t = month_start(cm_open(ds_id, 150, 151).time.values); COVER[ds_id] = (t.min(), t.max())
    except Exception:
        pass
print({lab: f"{COVER[i][0]:%Y-%m} -> {COVER[i][1]:%Y-%m}" for i, lab in GLORYS if i in COVER})

def cm_postprocess(parts):
    out = [da.weighted(np.cos(np.deg2rad(da.latitude))).mean("latitude") for da in parts]
    da = xr.concat(out, "longitude")
    da = da.assign_coords(longitude=da.longitude % 360).sortby("longitude").drop_duplicates("longitude")
    n = max(1, int(round(1.0 / float(np.median(np.diff(da.longitude.values))))))
    da = da.coarsen(longitude=n, boundary="trim").mean().load().rename({"longitude": "lon", "depth": "level"})
    da["time"] = month_start(da.time.values)
    return da

def glorys_year(y):
    f = f"{CACHE}/glorys_eq_{y}.nc"
    if os.path.exists(f) and y <= this_year - 2:
        return xr.open_dataarray(f).load()
    want = pd.date_range(f"{y}-01-01", f"{y}-12-01", freq="MS"); want = want[want <= TODAY]
    pieces = []
    for ds_id, label in GLORYS:
        if ds_id not in COVER or len(want) == 0: continue
        a, b = COVER[ds_id]; sel = want[(want >= a) & (want <= b)]
        if len(sel) == 0: continue
        t1 = sel[-1] + pd.offsets.MonthEnd(0)
        for k in range(3):
            try:
                da = cm_postprocess([cm_open(ds_id, 120, 180, sel[0], t1), cm_open(ds_id, -180, -80, sel[0], t1)]); break
            except Exception as e:
                if k == 2: raise
                time.sleep(20)
        pieces.append(da.assign_coords(source=("time", [label] * da.sizes["time"])))
        want = want.difference(sel)
    da = clean_time(xr.concat(pieces, "time")).rename("temp")
    if y <= this_year - 2:
        da.to_netcdf(f)
    print(f"  GLORYS {y}: {da.sizes['time']} months")
    return da

def d20(temp):
    T = temp.transpose(..., "level").values; z = temp.level.values
    below = T < 20
    k = np.clip(np.argmax(below, axis=-1), 1, len(z) - 1)
    t1 = np.take_along_axis(T, (k - 1)[..., None], -1)[..., 0]
    t2 = np.take_along_axis(T, k[..., None], -1)[..., 0]
    out = z[k - 1] + (20 - t1) * (z[k] - z[k - 1]) / (t2 - t1)
    out[~below.any(axis=-1)] = np.nan
    dims = [d for d in temp.transpose(..., "level").dims if d != "level"]
    return xr.DataArray(out, dims=dims, coords={d: temp[d] for d in dims}, name="d20")

T_all = clean_time(xr.concat([glorys_year(y) for y in range(1993, this_year + 1)], "time"))
D20 = d20(T_all)
D20_anom = D20.groupby("time.month") - D20.sel(time=slice("1993", "2020")).groupby("time.month").mean()
heat = pd.DataFrame({
    "d20_basin": D20_anom.sel(lon=slice(LON0, LON1)).mean("lon").to_series(),     # WWV proxy (recharge)
    "d20_west":  D20_anom.sel(lon=slice(120, 200)).mean("lon").to_series(),       # 120E-160W
    "d20_east":  D20_anom.sel(lon=slice(200, 280)).mean("lon").to_series(),       # 160W-80W
})
heat.index = month_start(heat.index)
heat.round(1).to_csv(f"{WEB}/heat_content_glorys.csv", index_label="date")
heat.tail(4).round(1)

## C. Trigger (H2): westerly wind bursts before each onset — ERA5
Window: **1 Nov (Y-1) → 30 Jun (Y)**. Same detection as Notebook 02 (≥5 m/s over ≥10° longitude, 130°E–150°W).
Accumulated westerly forcing = Σ over days of the mean positive wind anomaly across 130°E–160°W (m/s·day).

In [ ]:
import cdsapi
cds = cdsapi.Client(quiet=True)
GRID = [1.0, 1.0]
NC_LOCK = threading.Lock()

def process_era5(path, short):
    ds = xr.open_dataset(path)
    ds = ds.rename({k: v for k, v in {"valid_time": "time", "latitude": "lat", "longitude": "lon"}.items()
                    if k in ds.variables})
    if "expver" in ds.dims: ds = ds.mean("expver")
    da = ds["u10"]
    da = da.assign_coords(lon=da.lon % 360).sortby("lon").drop_duplicates("lon").sortby("lat").sel(lon=slice(LON0, LON1))
    da = da.weighted(np.cos(np.deg2rad(da.lat))).mean("lat").resample(time="1D").mean()
    out = da.rename(short).load(); ds.close()
    return out

def era5_chunk(short, year, months, days):
    tag = f"{short}_{year}_m{months[0]:02d}-{months[-1]:02d}_d{days[-1]:02d}"
    f = f"{CACHE}/era5_{tag}.nc"
    if os.path.exists(f):
        with NC_LOCK:
            return xr.open_dataarray(f).load()
    print(f"  requesting ERA5 {tag} (CDS queue, a few minutes)...")
    raw = f"/content/_era5_{tag}.nc"
    req = {"product_type": ["reanalysis"], "variable": ["10m_u_component_of_wind"], "year": [str(year)],
           "month": [f"{m:02d}" for m in months], "day": [f"{d:02d}" for d in days],
           "time": ["00:00", "06:00", "12:00", "18:00"], "area": [5, LON0, -5, LON1], "grid": GRID,
           "data_format": "netcdf", "download_format": "unarchived"}
    for k in range(3):
        try:
            cds.retrieve("reanalysis-era5-single-levels", req).download(raw); break
        except Exception as e:
            if k == 2: raise
            print("   retry:", str(e)[:80]); time.sleep(60)
    with NC_LOCK:
        da = process_era5(raw, short); os.remove(raw)
        da.to_netcdf(f + ".tmp"); os.replace(f + ".tmp", f)
    return da

def wind_year(y):
    cutoff = TODAY - pd.Timedelta(days=6)
    if y < cutoff.year:
        return era5_chunk("u", y, list(range(1, 13)), list(range(1, 32)))
    return era5_chunk("u", y, list(range(1, cutoff.month)), list(range(1, 32)))   # complete months (cached in NB02)

def daily_anom(da_now, da_clim):
    doy = da_clim.groupby("time.dayofyear").mean()
    doy = xr.concat([doy.isel(dayofyear=slice(-15, None)), doy, doy.isel(dayofyear=slice(0, 15))], "dayofyear")
    doy = doy.rolling(dayofyear=31, center=True).mean().isel(dayofyear=slice(15, -15))
    return (da_now.groupby("time.dayofyear") - doy).drop_vars("dayofyear")

def detect_wwb(ua, thresh=5.0, min_width=10.0, min_days=2, lon_range=(130, 210)):
    sub = ua.sel(lon=slice(*lon_range)); dlon = float(np.diff(sub.lon.values).mean()); rows = []
    for t in range(sub.sizes["time"]):
        v = sub.isel(time=t).values; best, run, start, span = 0, 0, 0, None
        for i, m in enumerate(np.append(v >= thresh, False)):
            if m:
                if run == 0: start = i
                run += 1
            else:
                if run > best: best, span = run, (start, i - 1)
                run = 0
        if best * dlon >= min_width:
            i0, i1 = span
            rows.append((pd.Timestamp(sub.time.values[t]), float(v[i0:i1+1].max()), float(sub.lon.values[i0:i1+1].mean())))
    if not rows:
        return pd.DataFrame(columns=["start", "end", "days", "peak_ms", "center_lon"])
    d = pd.DataFrame(rows, columns=["date", "peak", "lon"])
    d["event"] = (d.date.diff().dt.days.fillna(99) > 2).cumsum()
    ev = d.groupby("event").agg(start=("date", "min"), end=("date", "max"), peak_ms=("peak", "max"), center_lon=("lon", "mean"))
    ev["days"] = (ev.end - ev.start).dt.days + 1
    return ev[ev.days >= min_days][["start", "end", "days", "peak_ms", "center_lon"]].reset_index(drop=True)

print("Loading ERA5 wind (1991-2020 from cache; 2022-2023 requested once if missing)...")
u_clim = clean_time(xr.concat([wind_year(y) for y in range(1991, 2021)], "time"))
need = sorted({y for e in EVENTS for y in (e - 1, e)})
u_ev = clean_time(xr.concat([wind_year(y) for y in need], "time"))
ua_ev = daily_anom(u_ev, u_clim)

trig, wwb_all = {}, []
for e in EVENTS:
    w = ua_ev.sel(time=slice(f"{e-1}-11-01", f"{e}-06-30"))
    ev = detect_wwb(w); ev["event"] = e; wwb_all.append(ev)
    forcing = float(w.sel(lon=slice(130, 200)).clip(min=0).mean("lon").sum())
    trig[e] = {"wwb_events": len(ev), "wwb_days": int(ev.days.sum()) if len(ev) else 0,
               "wwb_peak_ms": float(ev.peak_ms.max()) if len(ev) else np.nan,
               "westerly_forcing_ms_day": round(forcing, 0)}
wwb_all = pd.concat(wwb_all, ignore_index=True)
wwb_all.to_csv(f"{WEB}/wwb_events_by_event.csv", index=False)
pd.DataFrame(trig).T

## D. Indonesia impact — CHIRPS rainfall, 1 Jun → same day-of-year as the latest 2026 data
National (land) mean anomaly vs 1991–2020, for each event year.

In [ ]:
chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/PENTAD").select("precipitation")
indo = ee.FeatureCollection("FAO/GAUL/2015/level0").filter(ee.Filter.eq("ADM0_NAME", "Indonesia"))
c_last = ee.Date(chirps.aggregate_max("system:time_start")).advance(5, "day")
NDAYS = c_last.difference(ee.Date(f"{this_year}-06-01"), "day").getInfo()
print(f"Rain window: 1 Jun -> +{NDAYS:.0f} days (to {c_last.format('YYYY-MM-dd').getInfo()})")

def window_total(year):
    s = ee.Date.fromYMD(year, 6, 1)
    return chirps.filterDate(s, s.advance(NDAYS, "day")).sum()

clim = ee.ImageCollection([window_total(y) for y in range(1991, 2021)]).mean()
def nat_anom(year):
    a = window_total(year).subtract(clim).divide(clim).multiply(100).updateMask(clim.gte(50))
    return a.reduceRegion(ee.Reducer.mean(), indo.geometry(), scale=25000, maxPixels=1e10, tileScale=4).get("precipitation")

rain = {e: round(ee.Number(nat_anom(e)).getInfo(), 1) for e in EVENTS}
print("Indonesia rainfall anomaly (%), Jun -> window end:", rain)

## E. Event comparison — aligned time series

In [ ]:
def aligned(series, e, m0=-12, m1=14):
    # Months relative to January of the onset year (0 = Jan Y).
    out = {}
    for k in range(m0, m1 + 1):
        t = pd.Timestamp(e, 1, 1) + pd.DateOffset(months=k)
        if t in series.index and pd.notna(series.loc[t]):
            out[k] = float(series.loc[t])
    return pd.Series(out)

panels = [("d20_basin", heat.d20_basin, "Recharge: equatorial D20 anomaly (m)"),
          ("nino34", idx.nino34, "Niño-3.4 (°C)"),
          ("nino12", idx.nino12, "Niño-1+2 — South American coast (°C)"),
          ("roni", idx.roni, "RONI — El Niño relative to tropical mean (°C)"),
          ("dmi", idx.dmi, "Indian Ocean Dipole (DMI, °C)"),
          ("bg", idx.oni - idx.roni, "ONI − RONI: background warming (°C)")]
fig, axs = plt.subplots(2, 3, figsize=(17, 8.5)); axs = axs.ravel()
ticks = list(range(-12, 15, 3))
labels = [(pd.Timestamp(2001, 1, 1) + pd.DateOffset(months=k)).strftime("%b") + ("\nY-1" if k < 0 else ("\nY" if k < 12 else "\nY+1")) for k in ticks]
event_json = {}
for a, (key, s, title) in zip(axs, panels):
    for e in EVENTS:
        al = aligned(s, e)
        event_json.setdefault(key, {})[str(e)] = {int(k): round(v, 3) for k, v in al.items()}
        a.plot(al.index, al.values, c=COLORS[e], lw=3 if e == 2026 else 1.6, label=f"{e}/{str(e+1)[2:]}")
    a.axhline(0, c="k", lw=.6); a.axvline(0, c="grey", lw=.6, ls=":")
    a.set_xticks(ticks); a.set_xticklabels(labels, fontsize=7); a.set_title(title, loc="left", fontweight="bold")
axs[0].legend()
fig.suptitle("Four El Niños aligned on January of the onset year", fontsize=13, fontweight="bold", x=.01, ha="left")
plt.tight_layout(); plt.savefig("events_aligned.png", dpi=200, bbox_inches="tight"); plt.show()
json.dump(event_json, open(f"{WEB}/events_aligned.json", "w"))

## F. Scorecard — what made 2026 different?

In [ ]:
def window(s, a, b):
    return s[(s.index >= pd.Timestamp(a)) & (s.index <= pd.Timestamp(b))].dropna()

score = {}
for e in EVENTS:
    pk = window(idx.oni, f"{e}-06-01", f"{e+1}-03-01")
    t_pk = pk.idxmax() if len(pk) else pd.NaT
    rech = window(heat.d20_basin, f"{e-1}-01-01", f"{e}-03-01")
    son = window(idx.dmi, f"{e}-09-01", f"{e}-11-01")
    score[e] = {
        "recharge_D20_max_m":        round(rech.max(), 1) if len(rech) else np.nan,
        "wwb_events_NovJun":         trig[e]["wwb_events"],
        "wwb_days_NovJun":           trig[e]["wwb_days"],
        "wwb_peak_ms":               round(trig[e]["wwb_peak_ms"], 1),
        "westerly_forcing_ms_day":   trig[e]["westerly_forcing_ms_day"],
        "peak_nino34_C":             round(window(idx.nino34, f"{e}-06-01", f"{e+1}-03-01").max(), 2),
        "peak_nino12_C":             round(window(idx.nino12, f"{e}-03-01", f"{e+1}-03-01").max(), 2),
        "peak_ONI_C":                round(pk.max(), 2) if len(pk) else np.nan,
        "RONI_at_peak_C":            round(idx.roni.get(t_pk, np.nan), 2),
        "background_warming_C":      round(idx.oni.get(t_pk, np.nan) - idx.roni.get(t_pk, np.nan), 2),
        "tropical_mean_at_peak_C":   round(idx.trop.get(t_pk, np.nan), 2),
        "DMI_SON_C":                 round(son.mean(), 2) if len(son) else np.nan,
        "indonesia_rain_anom_pct":   rain[e],
        "peak_month":                f"{t_pk:%Y-%m}" if pd.notna(t_pk) else "",
    }
sc = pd.DataFrame(score)
sc.columns = [f"{e}/{str(e+1)[2:]}" + (" (to date)" if e == this_year else "") for e in EVENTS]
sc.to_csv(f"{WEB}/scorecard.csv")
sc

In [ ]:
# Visual scorecard: each ingredient scaled to the strongest of the four events (1.0 = strongest)
items = {"recharge_D20_max_m": "Recharge\n(heat content)", "westerly_forcing_ms_day": "Trigger\n(westerly forcing)",
         "peak_nino12_C": "Coastal peak\n(Niño-1+2)", "peak_ONI_C": "Basin peak\n(ONI)",
         "background_warming_C": "Background\nwarming", "DMI_SON_C": "Positive IOD\n(Sep–Nov)"}
num = pd.DataFrame(score).T[list(items)].astype(float)
rel = num.clip(lower=0) / num.clip(lower=0).max()
fig, ax = plt.subplots(figsize=(13, 4.5)); w = .2
for i, e in enumerate(EVENTS):
    ax.bar(np.arange(len(items)) + (i - 1.5) * w, rel.loc[e], w, color=COLORS[e],
           label=f"{e}/{str(e+1)[2:]}" + (" (to date)" if e == this_year else ""))
ax.set_xticks(range(len(items))); ax.set_xticklabels(items.values())
ax.set_ylabel("relative to strongest event"); ax.set_ylim(0, 1.1); ax.legend(ncol=4, loc="upper center")
ax.set_title("Ingredients of four El Niños (negative values shown as 0)", loc="left", fontweight="bold")
plt.tight_layout(); plt.savefig("scorecard.png", dpi=200, bbox_inches="tight"); plt.show()

In [ ]:
# Auto-generated summary for the le-gis.com story page (edit freely before publishing)
s26, s97, s15 = score[2026], score[1997], score[2015]
def rank(key):
    v = pd.Series({e: score[e][key] for e in EVENTS}).astype(float)
    return int(v.rank(ascending=False)[2026])
lines = [
    "# Why 2026 became a historic El Niño — data summary",
    f"_Generated {TODAY:%d %b %Y} from OISST, ERA5, GLORYS12 and CHIRPS (1991–2020 base). 2026 values are to date._",
    "",
    f"- **Recharge:** pre-onset equatorial heat content peaked at {s26['recharge_D20_max_m']} m D20 anomaly "
    f"(rank {rank('recharge_D20_max_m')} of 4; 1997: {s97['recharge_D20_max_m']} m, 2015: {s15['recharge_D20_max_m']} m).",
    f"- **Trigger:** {s26['wwb_events_NovJun']} westerly wind bursts ({s26['wwb_days_NovJun']} days, peak {s26['wwb_peak_ms']} m/s) "
    f"between Nov 2025 and Jun 2026; accumulated westerly forcing {s26['westerly_forcing_ms_day']:.0f} m/s·day "
    f"(rank {rank('westerly_forcing_ms_day')} of 4).",
    f"- **Peak so far:** Niño-3.4 {s26['peak_nino34_C']} °C, Niño-1+2 {s26['peak_nino12_C']} °C, ONI {s26['peak_ONI_C']} °C "
    f"(1997/98 peak ONI: {s97['peak_ONI_C']} °C; 2015/16: {s15['peak_ONI_C']} °C).",
    f"- **Background warming:** ONI − RONI = {s26['background_warming_C']} °C at the latest peak "
    f"(1997: {s97['background_warming_C']} °C) — the share of the raw index that reflects a warmer tropical ocean overall.",
    f"- **Indian Ocean:** Sep–Nov DMI {s26['DMI_SON_C']} °C (1997: {s97['DMI_SON_C']} °C, 2015: {s15['DMI_SON_C']} °C).",
    f"- **Indonesia:** June–October rainfall {s26['indonesia_rain_anom_pct']} % vs normal "
    f"(1997: {s97['indonesia_rain_anom_pct']} %, 2015: {s15['indonesia_rain_anom_pct']} %, 2023: {score[2023]['indonesia_rain_anom_pct']} %).",
]
open(f"{WEB}/summary.md", "w").write("\n".join(lines))
os.system(f"cp events_aligned.png scorecard.png {WEB}/")
print("\n".join(lines)); print("\nSaved web package to", WEB)

## Interpretation guide
Fill in after running — compare 2026 with 1997 and 2015 on each ingredient:
1. **Recharge (H1)** — was more heat stored before onset? (Wyrtki 1975; Meinen & McPhaden 2000)
2. **Trigger (H2)** — stronger / earlier wind bursts? (McPhaden 1999 for 1997; Levine & McPhaden 2016 for the 2014 "failed" event)
3. **Coupling (H3)** — did the event go coastal first (eastern-Pacific type) like 1997, or central like 2015?
4. **Background (H4)** — how much of 2026's record raw index is the warmer ocean? RONI answers "how strong relative to its surroundings" (van Oldenborgh et al. 2021; L'Heureux et al. 2024).
5. **IOD** — a positive IOD in Sep–Nov compounds Indonesian drought (Saji et al. 1999; Ashok et al. 2001).

**Web package** (`MyDrive/le-gis_enso/web`): `indices_monthly_oisst_9120.csv`, `heat_content_glorys.csv`, `wwb_events_by_event.csv`,
`events_aligned.json`, `scorecard.csv`, `summary.md`, PNG figures — plus Notebook 02 outputs in `notebook02/`.

**References:** Bjerknes 1969; Wyrtki 1975; Saji et al. 1999 *Nature*; McPhaden 1999 *Science*; Meinen & McPhaden 2000 *J. Climate*;
Ashok et al. 2001 *GRL*; Levine & McPhaden 2016 *GRL*; Timmermann et al. 2018 *Nature*; van Oldenborgh et al. 2021 *ERL*;
Hersbach et al. 2020 *QJRMS* (ERA5); Lellouche et al. 2021 *Front. Earth Sci.* (GLORYS12); Huang et al. 2021 *J. Climate* (OISST v2.1); Funk et al. 2015 *Sci. Data* (CHIRPS).